# Question 1 - 88. Merge Sorted Array

You are given two integer arrays `nums1` and `nums2`, sorted in **non-decreasing order**, and two integers `m` and `n`, representing the number of elements in `nums1` and `nums2` respectively.

**Merge** `nums1` and `nums2` into a single array sorted in **non-decreasing order**.

The final sorted array should not be returned by the function, but instead be *stored inside the array* `nums1`. To accommodate this, `nums1` has a length of `m + n`, where the first `m` elements denote the elements that should be merged, and the last `n` elements are set to `0` and should be ignored. `nums2` has a length of `n`.

**Example 1:**

    Input: nums1 = [1,2,3,0,0,0], m = 3, nums2 = [2,5,6], n = 3
    Output: [1,2,2,3,5,6]
    Explanation: The arrays we are merging are [1,2,3] and [2,5,6].
    The result of the merge is [1,2,2,3,5,6].

**Example 2:**

    Input: nums1 = [1], m = 1, nums2 = [], n = 0
    Output: [1]
    Explanation: The arrays we are merging are [1] and [].

**Example 3:**

    Input: nums1 = [0], m = 0, nums2 = [1], n = 1
    Output: [1]
    Explanation: The arrays we are merging are [] and [1].
    Note that because m = 0, there are no elements in nums1. The 0 is only there to ensure the merge result can fit in nums1.

**Constraints:**

- `nums1.length == m + n`
- `nums2.length == n`
- `0 <= m, n <= 200`
- `1 <= m + n <= 200`
- `-10^9 <= nums1[i], nums2[j] <= 10^9`

**Follow up:** Can you come up with an algorithm that runs in `O(m + n)` time?

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers, filling from the back

📘 **Revise first:** [Part 3.4–3.5 · adding & removing costs](./0-concepts-array-and-string.ipynb) · [Pattern B · two pointers](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (copy in, then sort) | O((m+n) log(m+n)) | O(1)* |
| 2️⃣ Optimized (merge from the back) | O(m+n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You have **two piles of numbered cards, each already sorted** from small to large. You need to combine them into **one sorted pile**.

The catch: you're not allowed to grab a new table. Pile 1 (`nums1`) sits on a shelf that has **exactly enough empty slots at its end** to fit pile 2 (`nums2`). Those empty slots are shown as `0`s, which are placeholders, not real values. You must end up with the merged result on that shelf.

```
nums1 = [1, 2, 3, _, _, _]    ← 3 real cards + 3 empty slots
nums2 = [2, 5, 6]
goal  = [1, 2, 2, 3, 5, 6]
```

**Everyday picture:** two lines of people, each already sorted by height, need to merge into one sorted line in a hallway. There's empty space only at the **back** of line 1. The smart move is to start from the **tallest** people. Whoever is tallest across both lines walks to the very last spot. Then the next tallest takes the spot before that, and so on. Nobody ever has to stand where someone else is still standing.

### Step 0 · Clarify before coding

🗣️ *"Before I code, let me confirm the setup. Both arrays are sorted. nums1 has length m plus n, where the first m are real values and the last n are just space for me to use. I don't return anything; I modify nums1 in place."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Does `nums1` have exactly `n` spare slots at the end? | **Yes** (`len(nums1) == m + n`). | That free space is the key to the optimal solution. |
| Return a new array, or modify `nums1`? | **Modify in place**, return nothing. | Rules out "just build a new list and return it". |
| Duplicates? Negatives? | **Yes**, both possible. | Ties must be handled. Either choice is fine as long as nothing is lost. |
| What if `n == 0` or `m == 0`? | Both are allowed. | `n = 0` → nothing to do. `m = 0` → just copy `nums2` in. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** pour pile 2 into the empty slots, then ask the computer to **sort everything**.

🗣️ *"The simplest correct thing is to copy nums2 into the empty slots at the end of nums1 and then sort nums1. It's two lines and obviously correct."*

**Why isn't that good enough?** Sorting from scratch costs about **(m+n) × log(m+n)** steps. The `log` part is the price of a general-purpose sort that knows nothing about the input. But our input isn't random: **both halves are already sorted**. Calling sort throws that information away and redoes work that's already been done.

🗣️ *"That's O((m+n) log(m+n)) because of the sort. It ignores the fact that both inputs are already sorted. Merging two sorted lists should only take one pass, so let me use that."*

\* Python's built-in sort (Timsort) may use up to O(m+n) helper memory internally.

In [1]:
class SolutionBrute:
    def merge(self, nums1: list[int], m: int, nums2: list[int], n: int) -> None:
        nums1[m:] = nums2   # drop nums2 into the spare slots
        nums1.sort()        # re-sort everything from scratch

### Step 2 · Optimize: the "aha" moment

**Merging two sorted lists** is a classic: look at the front of each list, take the smaller one, repeat. One pass, O(m+n).

**But there's a trap if we merge from the front.** Watch:

```
nums1 = [1, 2, 3, _, _, _]     nums2 = [2, 5, 6]
```
Front-to-back, the smallest is `1` (already in place). Next smallest is `2` from nums1. Fine. Then `2` from nums2 must go into slot 2, **but slot 2 holds the `3` we haven't placed yet!** We'd overwrite it. Fixing that means shifting everything right, which brings back the slowness.

> 💡 **Aha:** the free space is at the **back**. So merge **from the back**, placing the **largest** value first. The slot we write into is always either empty or a value we've already moved. We can never overwrite something we still need.

**Three markers (pointers = index variables that remember a position):**
- `i` → the last *real* value in `nums1` (starts at `m − 1`)
- `j` → the last value in `nums2` (starts at `n − 1`)
- `k` → the slot we're filling next, from the very end (starts at `m + n − 1`)

At each step: compare `nums1[i]` and `nums2[j]`, copy the **bigger** one into `nums1[k]`, and step that pointer and `k` one to the left.

**When do we stop?** When `nums2` runs out (`j < 0`). Whatever is left of `nums1` is **already sitting in the right place**, because it's the smallest stuff and it never moved.

🗣️ *"Both arrays are sorted, so I can merge in a single pass. The danger with merging from the front is overwriting nums1 values I haven't placed yet. But the empty space is at the end, so I'll merge from the back: compare the largest remaining values of each array, put the bigger one in the last open slot, and move inward. The write position is always at or behind anything I still need to read, so nothing gets clobbered. I loop until nums2 is used up; any leftover nums1 values are already in place."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Copy in + `sort()` | ✅ quick first answer | Correct, but re-sorts data that's already sorted. O((m+n) log(m+n)). |
| Merge **front-to-back** into a **new** array, copy back | ⚠️ | Linear time, but needs a second array of size m+n. The problem hints we shouldn't. |
| Merge front-to-back **in place**, shifting to make room | ❌ | Each insertion shifts everything after it, which is O(m·n) in the worst case. |
| Use a heap (priority queue) | ❌ | A heap is good for merging *many* lists. For two, it just adds a `log` factor and extra memory. |
| **Merge back-to-front with three pointers** | ✅ **best** | Linear time, no extra memory, and it uses the free space exactly where it is. |

**Takeaway pattern:** *"when the free space is at the end, fill from the end."*

In [2]:
class Solution:
    def merge(self, nums1: list[int], m: int, nums2: list[int], n: int) -> None:
        i = m - 1          # last real value in nums1
        j = n - 1          # last value in nums2
        k = m + n - 1      # next slot to fill, from the back
        while j >= 0:      # only nums2 values can still be out of place
            if i >= 0 and nums1[i] > nums2[j]:
                nums1[k] = nums1[i]     # nums1's value is bigger: move it back
                i -= 1
            else:
                nums1[k] = nums2[j]     # nums2's value is bigger (or tie)
                j -= 1
            k -= 1

### Step 3 · Toy example walkthrough (tell it like a story)

`nums1 = [1, 2, 3, _, _, _]`, `m = 3`, `nums2 = [2, 5, 6]`, `n = 3`

| step | nums1[i] | nums2[j] | bigger? | write into slot k | nums1 afterwards |
|---|---|---|---|---|---|
| 1 | 3 | 6 | 6 | k=5 ← 6 | [1, 2, 3, _, _, **6**] |
| 2 | 3 | 5 | 5 | k=4 ← 5 | [1, 2, 3, _, **5**, 6] |
| 3 | 3 | 2 | **3** | k=3 ← 3 | [1, 2, 3, **3**, 5, 6] |
| 4 | 2 | 2 | tie → take nums2 | k=2 ← 2 | [1, 2, **2**, 3, 5, 6] |
| stop | | j < 0 | | | **[1, 2, 2, 3, 5, 6]** ✅ |

**Narrated:** "The biggest cards are 3 and 6. 6 wins, so it goes to the last slot. Next, 3 versus 5: 5 goes to the second-to-last slot. Now 3 versus 2: 3 is bigger, so the 3 moves back into slot 3. Then 2 versus 2, a tie; I take nums2's 2. Now nums2 is empty, and the leftover `1, 2` at the front of nums1 were already in the right spots, so I'm done."

**Why nothing gets overwritten:** the write slot `k` is always at or to the right of the read slot `i`. There are exactly `j + 1` "extra" slots between them, the room reserved for nums2's remaining cards.

In [3]:
def run(cls, nums1, m, nums2, n):
    nums1 = nums1[:]                 # copy so each test starts fresh
    cls().merge(nums1, m, nums2, n)
    return nums1

cases = [
    (([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3), [1, 2, 2, 3, 5, 6]),
    (([1], 1, [], 0), [1]),                                         # nothing to merge
    (([0], 0, [1], 1), [1]),                                        # nums1 empty
    (([4, 5, 6, 0, 0, 0], 3, [1, 2, 3], 3), [1, 2, 3, 4, 5, 6]),    # all nums2 smaller
    (([-1, 0, 0, 0], 1, [-3, -2, 5], 3), [-3, -2, -1, 5]),          # negatives
]
for args, expected in cases:
    assert run(SolutionBrute, *args) == expected
    assert run(Solution, *args) == expected
    print(args, "->", expected)

import random
for _ in range(500):
    a = sorted(random.randint(-5, 5) for _ in range(random.randint(0, 6)))
    b = sorted(random.randint(-5, 5) for _ in range(random.randint(0, 6)))
    if not a and not b:
        continue
    assert run(Solution, a + [0] * len(b), len(a), b, len(b)) == sorted(a + b)
print("All tests passed ✅")

([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3) -> [1, 2, 2, 3, 5, 6]
([1], 1, [], 0) -> [1]
([0], 0, [1], 1) -> [1]
([4, 5, 6, 0, 0, 0], 3, [1, 2, 3], 3) -> [1, 2, 3, 4, 5, 6]
([-1, 0, 0, 0], 1, [-3, -2, 5], 3) -> [-3, -2, -1, 5]
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Copy + sort | O((m+n) log(m+n)) | O(1)* | Re-sorts everything, ignoring that it was already sorted. |
| **Merge from the back** | **O(m+n)** | **O(1)** | Every value is moved at most once, and we store only three index numbers. |

**Why O(m+n) time?** Each loop iteration fills exactly one slot (`k` moves left by one), and there are at most m+n slots. So twice as much data means twice as much work, never more.

**Why O(1) memory?** We only keep `i`, `j`, `k`. Whether the arrays have 5 elements or 5 million, it's still three integers.

**Edge cases to mention:**
- `n = 0` → the loop never runs. Correct, nothing to do.
- `m = 0` → `i` starts at −1, so every step copies from nums2.
- All of nums2 is smaller than nums1 → nums1's values all slide to the back first, then nums2 fills the front.
- Ties → either choice keeps the result sorted.

**Likely follow-ups:**
- *"Why loop on `j` and not on `i`?"* → Leftover nums1 values are already in their final place, but leftover nums2 values still need copying.
- *"What if nums1 had no spare room?"* → Make a new array of size m+n and merge front-to-back. Still O(m+n) time, but O(m+n) memory.

---

#### 🗣️ Full interview script (about 60 seconds)

*"Both arrays are sorted, and nums1 has exactly n empty slots at the end for me to use.*

*The simplest solution is to copy nums2 into those slots and sort. That's correct but O((m+n) log(m+n)), and it wastes the fact that both halves are already sorted.*

*Since they're sorted, I can merge in one pass. Merging from the front would overwrite nums1 values I haven't placed yet, but the free space is at the back. So I'll merge from the back with three pointers: i at the last real value of nums1, j at the end of nums2, and k at the last slot. At each step I copy the larger of nums1[i] and nums2[j] into nums1[k] and move those pointers left. I stop when nums2 is exhausted, because whatever's left in nums1 is already in place.*

*Each slot is written once, so it's O(m+n) time, and I only use three index variables, so O(1) extra space."*